# 건강검진 코호트 EDA (10만 명 · 5년 · 분기별)

**이 노트북은 EDA만 한다. 모델 학습은 `colab_ml_transition.ipynb` 로 분리.**

목적: 어떤 지표가 실제로 예측력이 있는지 확인해서, 모델에 넣을 특성을 확정한다.

핵심 주의점
- `grade`(종합판정)는 검진 지표들을 합산해 만든 값이다. **`grade`와의 상관은 발견이 아니라 정의상 종속.**
- 특성 선택은 실제 타겟 — **"현재 정상·주의인 사람이 다음 분기에 위험으로 전환되는가"** — 기준으로 판단한다. (섹션 6)
- **연속 변수는 상관계수로, 이진 변수는 비율비로 판단한다.** (섹션 6-2) 이진 변수는 점이연 상관이 구조적으로 작게 나와서 상관만 보면 버리게 된다.

## 0. 환경 준비

In [ ]:
!pip -q install pyarrow
!apt-get install -y fonts-nanum > /dev/null 2>&1

import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns, warnings
import matplotlib.font_manager as fm
fm.fontManager.addfont('/usr/share/fonts/truetype/nanum/NanumGothic.ttf')
plt.rc('font', family='NanumGothic')
plt.rc('axes', unicode_minus=False)
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', font='NanumGothic')
pd.set_option('display.width', 200)
print('준비 완료')

## 1. 데이터 로드

In [ ]:
import os

PATH = '/content/checkups.parquet'   # 코랩 왼쪽 [파일] 탭에 드래그해서 올린 위치

assert os.path.exists(PATH), (
    '파일이 없음. 코랩 왼쪽 [파일] 탭(폴더 아이콘)에 checkups.parquet 을 드래그해서 '
    '업로드가 끝난 뒤 이 셀을 다시 실행하세요.')
print(f'파일 확인 {os.path.getsize(PATH)/1e6:.0f}MB')

df = pd.read_parquet(PATH)
print(f'{len(df):,} 행 · {df.person_id.nunique():,} 명 · {df.quarter.nunique()} 분기')

KOR = {
    'bmi':'체질량지수(BMI)', 'waist':'허리둘레', 'systolic':'수축기혈압',
    'diastolic':'이완기혈압', 'fbs':'공복혈당', 'total_chol':'총콜레스테롤',
    'triglyceride':'중성지방', 'hdl':'HDL(좋은콜)', 'ldl':'LDL(나쁜콜)',
    'hemoglobin':'혈색소', 'ast':'간수치AST', 'alt':'간수치ALT',
    'ggt':'감마지티피', 'creatinine':'크레아티닌',
    'age':'나이', 'grade':'종합판정', 'transition':'위험전환',
}
kn = lambda c: KOR.get(c, c)

METRICS = ['bmi','waist','systolic','diastolic','fbs','total_chol',
           'triglyceride','hdl','ldl','hemoglobin','ast','alt','ggt','creatinine']

def band(a):
    return ('20대이하' if a<30 else '30대' if a<40 else '40대' if a<50
            else '50대' if a<60 else '60대' if a<70 else '70대+')
ORDER = ['20대이하','30대','40대','50대','60대','70대+']
df.head()

## 2. 기본 정보 / 결측 / 수검 횟수

In [ ]:
print('=== 형태 ===');  print(df.shape)
print('\n=== 타입 ===');  print(df.dtypes)
print('\n=== 결측 ===');  print(df.isna().sum()[lambda s: s>0] if df.isna().any().any() else '없음 (미수검은 행 자체가 빠짐)')

print('\n=== 인원별 관측 수 ===')
cnt = df.groupby('person_id').size()
print(cnt.describe().round(2))
print(f'\n20회 미만(미수검 있음) 인원 비율: {(cnt<20).mean()*100:.1f}%')

df.describe().T.round(2)

## 3. 인구 분포 (연령·성별·흡연·판정)

> 나이 히스토그램은 **1세 단위(bins=66)** 로 그린다.
> `bins=30` 으로 그리면 bin 폭이 2.17세가 되는데 나이는 정수라서
> 어떤 bin은 정수 2개, 어떤 bin은 3개를 담아 **높이가 들쭉날쭉해 보인다.**
> 데이터 결함이 아니라 binning 아티팩트다. (섹션 3-2에서 실제 분포 검증)

In [ ]:
base = df[df.quarter == 0].copy()
base['연령대'] = base.age.map(band)

fig, ax = plt.subplots(2, 2, figsize=(14, 9))
sns.countplot(data=base, x='연령대', hue='sex', order=ORDER, ax=ax[0,0]); ax[0,0].set_title('연령대×성별 분포')
sns.countplot(data=df, x='grade', hue='sex', ax=ax[0,1]); ax[0,1].set_title('종합판정×성별 (0정상 1주의 2위험)')
sns.countplot(data=base, x='smoker', hue='sex', ax=ax[1,0]); ax[1,0].set_title('흡연×성별')
sns.histplot(data=base, x='age', hue='sex', bins=66, ax=ax[1,1]); ax[1,1].set_title('연령 분포 (1세 단위)')
plt.tight_layout(); plt.show()

print('판정 비율:', (df.grade.value_counts(normalize=True).sort_index()*100).round(1).to_dict(), '(목표 40.2 / 32.2 / 27.6)')
print('흡연율: 전체 %.1f%% / 남 %.1f%% / 여 %.1f%%' % (
    base.smoker.mean()*100, base[base.sex=='M'].smoker.mean()*100, base[base.sex=='F'].smoker.mean()*100))

### 3-2. 연령 분포가 목표를 재현했는지 검증 (스파이크 진위 확인)

In [ ]:
TARGET_RATIO = {'20대이하':0.154, '30대':0.157, '40대':0.212,
                '50대':0.226, '60대':0.185, '70대+':0.066}   # 통계연보 100만 기준

act = base.연령대.value_counts()
chk = pd.DataFrame({
    '실제인원': [act.get(k,0) for k in ORDER],
    '실제비율%': [act.get(k,0)/len(base)*100 for k in ORDER],
    '목표비율%': [TARGET_RATIO[k]*100 for k in ORDER],
}, index=ORDER)
chk['차이%p'] = (chk['실제비율%'] - chk['목표비율%']).round(2)
chk[['실제비율%','목표비율%']] = chk[['실제비율%','목표비율%']].round(2)
print('=== 연령대 분포 재현 검증 ===')
print(chk.to_string())
print(f"\n최대 오차 {chk['차이%p'].abs().max():.2f}%p  ->  {'재현 정확' if chk['차이%p'].abs().max()<0.5 else '재현 실패'}")
print(f'\n첫 분기 인원 {len(base):,}명 / 전체 {df.person_id.nunique():,}명'
      f' = {len(base)/df.person_id.nunique()*100:.1f}%  (미수검률 10%와 일치하면 정상)')

fig, ax = plt.subplots(1, 2, figsize=(14, 4))
vc = base.age.round().astype(int).value_counts().sort_index()
ax[0].bar(vc.index, vc.values, width=.9); ax[0].set_title('1세 단위 인원 (실제 분포)')
ax[0].set_xlabel('나이'); ax[0].set_ylabel('명')
chk[['실제비율%','목표비율%']].plot(kind='bar', ax=ax[1])
ax[1].set_title('연령대 비율 — 실제 vs 목표'); ax[1].set_xlabel(''); ax[1].tick_params(axis='x', rotation=0)
plt.tight_layout(); plt.show()

## 4. 검진 지표 분포 & 이상치

In [ ]:
fig, axes = plt.subplots(4, 4, figsize=(16, 12))
for ax, c in zip(axes.ravel(), METRICS):
    sns.histplot(df[c], bins=60, ax=ax)
    ax.set_title(f'{kn(c)}  [{c}]', fontsize=10); ax.set_xlabel('')
for ax in axes.ravel()[len(METRICS):]:
    ax.axis('off')
plt.tight_layout(); plt.show()

print('=== 이상치 비율 (IQR 1.5배 기준) ===')
print(f"  {'지표':<16}{'비율':>7}{'최소':>9}{'최대':>11}")
for c in METRICS:
    q1, q3 = df[c].quantile([.25, .75]); iqr = q3 - q1
    out = ((df[c] < q1-1.5*iqr) | (df[c] > q3+1.5*iqr)).mean()*100
    print(f'  {kn(c):<14} {out:5.2f}%  {df[c].min():8.1f}{df[c].max():11.1f}')

print()
print('해석 1) 중성지방·감마지티피·AST·ALT는 로그정규 분포라 IQR 기준 이상치가 원래 많이 잡힘')
print('        -> 실제 오류가 아님. 모델링에서 로그변환 대상.')
print('해석 2) 혈압 500대, 혈당 500대 같은 값은 생리학적으로 불가능 -> 주입한 측정오류.')
print('        -> 모델링에서 클리핑 대상.')

## 5. 상관관계 (지표끼리 / 겹치는 정보 확인)

In [ ]:
corr = df[METRICS + ['age','grade']].corr()
lab = [kn(c) for c in corr.columns]
plt.figure(figsize=(13, 10))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0, square=True,
            xticklabels=lab, yticklabels=lab, cbar_kws={'shrink':.8})
plt.title('지표 상관관계'); plt.tight_layout(); plt.show()

c_grade = corr['grade'].drop('grade').abs().sort_values(ascending=False)

print('=== 겹치는 정보 (지표끼리 |상관| >= 0.7) ===')
cm = corr.drop(index='grade', columns='grade')
for a in cm.columns:
    for b in cm.columns:
        if a < b and abs(cm.loc[a, b]) >= 0.7:
            print(f'  {kn(a)} <-> {kn(b)}   {cm.loc[a,b]:+.2f}   거의 같은 정보')

print()
print('!! 주의: grade(종합판정)는 지표들을 합산해 만든 값이다.')
print('   따라서 grade와의 상관은 "발견"이 아니라 정의상 종속이다.')
print('   -> 특성 선택 기준으로 쓰면 안 된다. 섹션 6에서 실제 타겟 기준으로 다시 본다.')

## 6. ★ 실제 예측 타겟(위험 전환)과의 상관 — 연속 변수 특성 선택

**타겟**: 현재 *정상·주의* 인 사람이 → **다음 분기에 위험으로 전환**되는가 (0/1)

- 이미 위험인 사람은 제외. 이미 아픈 사람 맞히는 건 임상 가치 없음.
- 전이 관성이 강해서(위험→위험 75.5%) 그냥 "다음 분기 위험 여부"를 맞히면
  모델이 **"지금 나쁘면 다음에도 나쁘다"** 만 학습한다. 그건 의사가 이미 안다.
- 위험군을 빼면 모델은 **변화·추세**를 봐야만 맞힐 수 있게 된다.

In [ ]:
# t+1 분기 판정을 t 행에 붙임 (실제로 다음 분기 검진이 있는 행만 남음)
nxt = df[['person_id','quarter','grade']].rename(columns={'grade':'next_grade'}).copy()
nxt['quarter'] -= 1
e = df.merge(nxt, on=['person_id','quarter'], how='inner')

elig = e[e.grade < 2].copy()                       # 현재 정상·주의만
elig['transition'] = (elig.next_grade == 2).astype(int)
elig['연령대'] = elig.age.map(band)
BASE = elig.transition.mean()

print(f'전체 {len(df):,}행 -> 다음분기 검진 있는 행 {len(e):,} -> 현재 정상·주의 {len(elig):,}')
print(f'양성률(위험 전환)  {BASE*100:.2f}%')
print()
print('현재 판정별 전환율:')
for gv, nm in [(0,'정상'), (1,'주의')]:
    sub = elig[elig.grade == gv]
    print(f'  {nm}({gv})  {len(sub):>9,}행  ->  위험 전환 {sub.transition.mean()*100:5.2f}%'
          f'  · 양성 {sub.transition.sum():>7,}개')
print()
pos = elig.groupby('grade').transition.sum()
print(f'전체 양성 {pos.sum():,}개 중 주의 출신 {pos[1]/pos.sum()*100:.1f}%')
print('-> 합쳐서 모델 하나로 학습하면 사실상 주의 집단만 학습하게 됨. 평가는 반드시 분리.')

In [ ]:
# ---- 두 타겟과의 상관 비교 ----
c_trans = elig[METRICS + ['age','transition']].corr()['transition'].drop('transition')

cmp = pd.DataFrame({
    'grade상관(정의상·참고)': c_grade.reindex(c_trans.index).round(3),
    '전환상관(실제타겟)':      c_trans.abs().round(3),
    '방향': np.where(c_trans >= 0, '↑높을수록위험', '↓낮을수록위험'),
})
cmp['거품비율'] = (cmp['전환상관(실제타겟)'] / cmp['grade상관(정의상·참고)']).round(2)
cmp = cmp.sort_values('전환상관(실제타겟)', ascending=False)
cmp.index = [kn(c) for c in cmp.index]

print('=== 연속 특성 예측력 (전환상관 내림차순) ===')
print(cmp.to_string())
print()
print('거품비율 = 전환상관 / grade상관.  낮을수록 grade 계산식에 직접 들어간 지표라')
print('grade 상관이 부풀려져 있었다는 뜻. (판정식 포함: 혈압·혈당·BMI·LDL·HDL·중성지방·허리)')

In [ ]:
# ---- 시각화 + 제외 후보 (연속 변수 한정) ----
THRESH = 0.05
o = c_trans.abs().sort_values()
plt.figure(figsize=(9, 6))
plt.barh([kn(i) for i in o.index], o.values,
         color=['#c0392b' if v >= .22 else '#e67e22' if v >= .15 else
                '#f1c40f' if v >= THRESH else '#bdc3c7' for v in o.values])
plt.axvline(THRESH, ls='--', c='k', lw=1)
plt.xlabel('|상관|  (다음 분기 위험 전환)')
plt.title(f'연속 특성 — 회색 = |상관| < {THRESH} (제외)')
plt.tight_layout(); plt.show()

WEAK = c_trans[c_trans.abs() < THRESH].index.tolist()
KEEP = [c for c in METRICS if c not in WEAK]
print(f'제외 ({len(WEAK)}개): {[kn(c) for c in WEAK] or "없음"}')
print(f'유지 ({len(KEEP)}개): {[kn(c) for c in KEEP]}')
print('\n※ 이 임계값은 연속 변수에만 적용한다. 이진 변수는 섹션 6-2에서 따로 판단.')

## 6-2. ★ 이진 변수(흡연·성별)는 **비율비**로 판단

이진 변수는 점이연 상관이 **구조적으로 작게 나온다.** 상관계수만 보고 자르면
실제로는 강한 위험인자인 흡연을 버리게 된다.

```
흡연   상관 r ≈ 0.06  ->  실제 전환율 1.5배 이상 차이   (강한 효과)
혈색소 상관 r ≈ 0.04  ->  실제 차이 거의 없음          (무효과)
```
숫자는 비슷한데 의미가 정반대. **이진 변수는 비율비(risk ratio)로 본다.**

In [ ]:
print(f'기준 전환율 {BASE*100:.2f}%\n')
for col, nm in [('smoker','흡연'), ('sex','성별')]:
    t = elig.groupby(elig[col].astype(str)).transition.agg(행수='size', 전환율='mean')
    t['전환율%'] = (t['전환율']*100).round(2); t = t.drop(columns='전환율')
    t['기준대비'] = (t['전환율%']/(BASE*100)).round(2)
    v = (elig[col] == elig[col].unique()[0]).astype(int)
    r = abs(np.corrcoef(v, elig.transition)[0, 1])
    hi, lo = t['전환율%'].max(), t['전환율%'].min()
    print(f'[{nm}]  상관 |r|={r:.3f}  ·  비율비 {hi/lo:.2f}배   <- 판단은 비율비로')
    print(t.to_string()); print()

print('교차 (성별 × 흡연) 전환율%:')
print((elig.pivot_table(index='sex', columns='smoker',
                        values='transition', aggfunc='mean')*100).round(2).to_string())

In [ ]:
# ---- 나이 교란 배제: 같은 연령대 안에서도 흡연 효과가 남는가 ----
fig, ax = plt.subplots(1, 2, figsize=(14, 4.5))
for i, sx in enumerate(['M', 'F']):
    t = (elig[elig.sex == sx].pivot_table(index='연령대', columns='smoker',
                                          values='transition', aggfunc='mean')*100).round(2)
    t = t.reindex(ORDER); t.columns = ['비흡연', '흡연']
    t.plot(kind='bar', ax=ax[i], color=['#7f8c8d', '#c0392b'])
    ax[i].set_title(f'{"남성" if sx=="M" else "여성"} — 연령대별 흡연 효과')
    ax[i].set_ylabel('위험 전환율(%)'); ax[i].set_xlabel(''); ax[i].tick_params(axis='x', rotation=0)
plt.tight_layout(); plt.show()

t = (elig[elig.sex=='M'].pivot_table(index='연령대', columns='smoker',
                                     values='transition', aggfunc='mean')*100).round(2)
t = t.reindex(ORDER); t.columns = ['비흡연','흡연']; t['차이%p'] = (t['흡연']-t['비흡연']).round(2)
print('남성 — 연령대별 흡연 효과')
print(t.to_string())
print('\n모든 연령대에서 차이가 같은 방향이면 나이 교란이 아니라 흡연 자체의 효과.')
BINARY_KEEP = ['smoker', 'sex']
print(f'\n-> 이진 특성 채택: {BINARY_KEEP}')

## 7. 시계열 추세 (5년) & 판정 전이 행렬

In [ ]:
q = df.groupby('quarter').agg(
    bmi=('bmi','mean'), systolic=('systolic','mean'), fbs=('fbs','mean'),
    ldl=('ldl','mean'), 위험비율=('grade', lambda s: (s==2).mean()*100)).reset_index()

fig, ax = plt.subplots(1, 2, figsize=(15, 4.5))
for c in ['bmi','systolic','fbs','ldl']:
    ax[0].plot(q.quarter, q[c]/q[c].iloc[0]*100, marker='o', label=kn(c))
ax[0].set_title('지표 평균 추이 (첫 분기 = 100)'); ax[0].set_xlabel('분기'); ax[0].legend()
ax[1].plot(q.quarter, q.위험비율, marker='o', color='crimson')
ax[1].set_title('위험 판정 비율(%) 추이'); ax[1].set_xlabel('분기')
plt.tight_layout(); plt.show()

print('5년(20분기) 변화율:')
for c in ['bmi','systolic','fbs','ldl']:
    print(f'  {kn(c):<14} {(q[c].iloc[-1]/q[c].iloc[0]-1)*100:+.2f}%')
print(f'  위험 판정 비율 {q.위험비율.iloc[0]:.1f}% -> {q.위험비율.iloc[-1]:.1f}%'
      f'  ({q.위험비율.iloc[-1]-q.위험비율.iloc[0]:+.1f}%p)')
print()

d = df.sort_values(['person_id','quarter'])
d['ng'] = d.groupby('person_id')['grade'].shift(-1)
trans = pd.crosstab(d.grade, d.ng, normalize='index').round(3)
trans.index = ['정상','주의','위험']; trans.columns = ['정상','주의','위험']
print('=== 판정 전이확률 (행 = 현재, 열 = 다음 분기) ===')
print(trans.to_string())

# 신규 위험 유입의 출처 (판정 분포 x 전이확률)
share = df.grade.value_counts(normalize=True).sort_index()
inflow = {'정상 출신': share[0]*trans.loc['정상','위험'],
          '주의 출신': share[1]*trans.loc['주의','위험']}
tot = sum(inflow.values())
print('\n=== 신규 위험 진입자의 출처 ===')
for k, v in inflow.items():
    print(f'  {k}  {v/tot*100:5.1f}%')
print('-> 위험까지 가려면 거의 반드시 주의를 거친다. 개입 지점은 주의 단계.')

plt.figure(figsize=(5.5, 4.5))
sns.heatmap(trans, annot=True, fmt='.3f', cmap='Blues', cbar=False)
plt.title('판정 전이확률'); plt.xlabel('다음 분기'); plt.ylabel('현재')
plt.tight_layout(); plt.show()

## 7-2. ★ 미수검 간격(gap) — 파생특성 오염 확인

미수검자는 **행 자체가 빠진다.** 그런데 `shift(1)` 은 "직전 **행**"을 가져오므로
직전 행이 1분기 전이라는 보장이 없다.

```
person A:  q0 → q1 → q2 → q3     d1 = 3개월치 변화
person B:  q0 → ✗  → ✗  → q3     d1 = 9개월치 변화   <- 같은 컬럼에 섞임
```
`d1`(변화량)의 **시간 단위가 뒤섞이므로 gap 으로 나눠 분기당 변화율로 통일**해야 한다.
gap 자체가 예측력이 있는지도 같이 확인한다.

In [ ]:
s = df.sort_values(['person_id','quarter']).copy()
s['gap'] = s.quarter - s.groupby('person_id')['quarter'].shift(1)
gg = s.dropna(subset=['gap'])

dist = (gg.gap.value_counts(normalize=True).sort_index()*100)
print('=== 직전 관측과의 간격 분포 ===')
for k, v in dist.items():
    tag = '정상' if k == 1 else 'lag1이 실제로는 이만큼 떨어짐'
    print(f'  {int(k)}분기 전  {v:6.2f}%   <- {tag}')
CONTAM = (gg.gap > 1).mean()*100
print(f'\ngap > 1 비율 = {CONTAM:.2f}%  <- 이만큼의 d1/ma4가 시간단위 오염됨')

# gap 자체의 예측력
sg = s.merge(nxt, on=['person_id','quarter'], how='inner')
sg = sg[(sg.grade < 2) & sg.gap.notna()].copy()
sg['tr'] = (sg.next_grade == 2).astype(int)
t = sg.groupby(sg.gap.clip(upper=3)).tr.agg(행수='size', 전환율='mean')
t['전환율%'] = (t['전환율']*100).round(2); t = t.drop(columns='전환율')
print('\n=== gap 자체의 예측력 ===')
print(t.to_string())

cnt2 = df.groupby('person_id').size().rename('n_obs')
sg2 = sg.merge(cnt2, on='person_id')
t2 = sg2.groupby(pd.cut(sg2.n_obs, [0,15,17,19,20],
                        labels=['~15회','16-17회','18-19회','20회(개근)']),
                 observed=True).tr.agg(행수='size', 전환율='mean')
t2['전환율%'] = (t2['전환율']*100).round(2); t2 = t2.drop(columns='전환율')
print('\n=== 개인 총 수검횟수와 전환율 ===')
print(t2.to_string())

spread = t['전환율%'].max() - t['전환율%'].min()
print(f'\ngap 구간 간 전환율 편차 {spread:.2f}%p (기준 {BASE*100:.2f}%)')
print('-> 편차가 작으면 gap 자체는 예측력 없음. 특성에서 제외하고 d1 보정용으로만 쓴다.')
print('   (생성기가 미수검을 건강상태와 무관하게 랜덤 적용했기 때문. 실제 데이터는 다를 수 있음 — 한계)')

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].bar(dist.index.astype(int), dist.values, color='#34495e')
ax[0].set_title('직전 관측과의 간격 분포'); ax[0].set_xlabel('분기'); ax[0].set_ylabel('%')
ax[1].bar(t.index.astype(str), t['전환율%'], color='#7f8c8d')
ax[1].axhline(BASE*100, ls='--', c='crimson', label=f'기준 {BASE*100:.2f}%')
ax[1].set_title('gap별 위험 전환율'); ax[1].set_xlabel('gap(분기)'); ax[1].set_ylabel('%'); ax[1].legend()
plt.tight_layout(); plt.show()

## 8. 연령대 · 성별 위험도

In [ ]:
df['연령대'] = df.age.map(band)
p = (df.groupby(['연령대','sex'])['grade'].apply(lambda s: (s==2).mean()*100)
       .unstack().round(1).reindex(ORDER))
pt = (elig.groupby(['연령대','sex'])['transition'].mean()
        .unstack().mul(100).round(2).reindex(ORDER))
print('=== 연령대×성별 위험 판정 비율(%) ===');  print(p.to_string())
print('\n=== 연령대×성별 위험 전환 비율(%) — 현재 정상·주의만 ===');  print(pt.to_string())

r1 = p.max().max()/p.min().min(); r2 = pt.max().max()/pt.min().min()
print(f'\n최고/최저 격차:  판정률 {r1:.1f}배  vs  전환율 {r2:.1f}배')
print('-> 전환은 나이로 덜 설명된다. 연령 필터만으로는 전환자를 못 거른다.')

fig, ax = plt.subplots(1, 2, figsize=(14, 4.5))
p.plot(kind='bar', ax=ax[0]); ax[0].set_title('위험 판정 비율(%)'); ax[0].set_ylabel('%')
pt.plot(kind='bar', ax=ax[1]); ax[1].set_title('위험 전환 비율(%) — 조기경보 대상'); ax[1].set_ylabel('%')
for a in ax: a.set_xlabel(''); a.tick_params(axis='x', rotation=0)
plt.tight_layout(); plt.show()

## 9. 개인 궤적 샘플 (시계열성 확인)

In [ ]:
ids = df.person_id.drop_duplicates().sample(6, random_state=0).tolist()
fig, axes = plt.subplots(2, 3, figsize=(16, 7))
for ax, pid in zip(axes.ravel(), ids):
    g = df[df.person_id==pid].sort_values('quarter')
    ax.plot(g.quarter, g.systolic, marker='o', label='수축기혈압')
    ax.plot(g.quarter, g.fbs, marker='s', label='공복혈당')
    ax2 = ax.twinx(); ax2.step(g.quarter, g.grade, color='gray', alpha=.5, where='mid')
    ax2.set_ylim(-0.2, 2.2); ax2.set_yticks([0,1,2]); ax2.set_yticklabels(['정상','주의','위험'])
    ax.set_title(f'person {pid} ({g.sex.iloc[0]}, {g.age.iloc[0]:.0f}세)')
    ax.set_xlabel('분기'); ax.legend(fontsize=8, loc='upper left')
plt.tight_layout(); plt.show()

chg = df.sort_values(['person_id','quarter']).groupby('person_id')['grade'].apply(
    lambda x: int((x.diff().fillna(0) != 0).sum()))
print(f'개인별 등급 변경 횟수  평균 {chg.mean():.1f}회 / 중앙 {chg.median():.0f}회')
print(f'5년간 한 번도 안 바뀐 사람 {(chg==0).mean()*100:.1f}%  ·  3회 이상 {(chg>=3).mean()*100:.1f}%')
print('-> 개인 내 변동이 크다 = 단일 시점 판정은 불안정 = 추적 관찰이 필요하다는 근거')

## 10. EDA 요약 — 모델링 사양 확정

In [ ]:
print('='*72)
print('EDA 요약 — 이 값들이 colab_ml_transition.ipynb 의 입력 사양이 된다')
print('='*72)
print(f'1) 규모      {len(df):,}행 · {df.person_id.nunique():,}명 · {df.quarter.nunique()}분기(5년)')
print(f'   판정 분포 {(df.grade.value_counts(normalize=True).sort_index()*100).round(1).tolist()} (정상/주의/위험)')
print(f'   결측      셀 결측 없음. 미수검은 행 자체가 빠짐 (20회 미만 {(cnt<20).mean()*100:.1f}%)')
print(f'   연령분포  목표 대비 최대 오차 {chk["차이%p"].abs().max():.2f}%p — 재현 정확')
print()
print('2) 예측 타겟  현재 정상·주의 -> 다음 분기 위험 전환')
print(f'   대상       {len(elig):,}행 · 양성률 {BASE*100:.2f}%')
print(f'   정상 {elig[elig.grade==0].transition.mean()*100:.2f}% / 주의 {elig[elig.grade==1].transition.mean()*100:.2f}%'
      f'  · 양성의 {pos[1]/pos.sum()*100:.1f}%가 주의 출신')
print(f'   신규 위험 진입자의 {inflow["주의 출신"]/tot*100:.1f}%가 직전 분기 주의')
print()
print(f'3) 연속 특성 ({len(KEEP)}개 + 나이) — 상관 기준')
for c in c_trans.abs().sort_values(ascending=False).index:
    if c in KEEP or c == 'age':
        print(f'   {kn(c):<16} {abs(c_trans[c]):.3f}')
print(f'   제외: {[kn(c) for c in WEAK]}  (상관 {THRESH} 미만)')
print()
print('4) 이진 특성 — 비율비 기준 (상관계수로 판단하면 안 됨)')
for col, nm in [('smoker','흡연'), ('sex','성별')]:
    t = elig.groupby(elig[col].astype(str)).transition.mean()*100
    print(f'   {nm:<6} {t.max()/t.min():.2f}배  ({t.min():.2f}% -> {t.max():.2f}%)')
print()
print('5) 모델링 사양')
print('   - 클리핑    혈압/혈당/중성지방/간수치 생리학적 상한 (주입된 측정오류 제거)')
print('   - 로그변환  중성지방, 감마지티피, AST, ALT (로그정규 분포)')
print(f'   - 파생특성  lag1 / d1(gap으로 나눠 분기당 변화율, 오염 {CONTAM:.1f}%) / ma4 / std4 / dev')
print('   - gap       특성으로는 제외 (예측력 없음). d1 보정용으로만 사용')
print('   - 제외      severity (데이터 생성기 내부 점수. 실제 병원에 없음)')
print('   - 분할      GroupShuffleSplit (사람 단위. 같은 사람이 train/test에 겹치면 누출)')
print('   - 평가      PR-AUC 주력 (기준선 %.3f), AUC 보조' % BASE)
print('   - 하위집단  정상 출발 / 주의 출발 성능을 반드시 분리 평가')
print('='*72)